# 📈 Notebook 5: Automated Dynamics Monitoring & Simulation

This notebook combines a **Programmatic Motion Generator** and a **High-Frequency Data Logger** to capture and analyze robot dynamics in a fully automated loop—no terminal inputs or widgets required

### 🎯 Objective
* **Auto-Drive**: Programmatically command Gazebo joints using a smooth Half-Sine (S-curve-like) trajectory generator.
* **Visualize**: Map actual joint velocity and torque (effort) feedback in real-time to analyze motor inertia and acceleration characteristics.

---

### 📋 Prerequisites
Before running, ensure that Gazebo and the Simulation Bridge are active in your background terminals:

1. **Terminal 1 (Gazebo Simulator)**:
   > ros2 launch movensys_manipulator_description gazebo_trajectory_simulation.launch.py
2. **Terminal 2 (Simulation Bridge)**:
   > ros2 launch movensys_manipulator_moveit_config sim_bridge.launch.py simulator:=gazebo use_sim_time:=true



## 📥 Step 1: Smooth Trajectory Generation & Telemetry Logging
We will subscribe to `/joint_states` at 100Hz while simultaneously publishing smooth joint commands to `/gazebo_position_controller/commands` at 50Hz. 
This code executes a **Half-Sine velocity curve** over a 4.0-second window, ensuring clean data logging without mechanical or computational shock.


In [ ]:
import rclpy
from wmx_utils import WmxClient
from sensor_msgs.msg import JointState
from std_msgs.msg import Float64MultiArray
import numpy as np
import time

# 1. Initialize ROS 2 Context & Fetch the centralized WmxClient Node
if not rclpy.ok():
    rclpy.init()

wmx = WmxClient(node_name='wmx_auto_dynamics_client_05')
print(f"✅ Auto-Dynamics Client Activated. Target Axes: {wmx.axis_list}")

# 2. Topic publishers and subscribers
pub_cmd = wmx.create_publisher(Float64MultiArray, '/gazebo_position_controller/commands', 10)

telemetry_buffer = []
is_logging = True

def joint_states_callback(msg):
    global telemetry_buffer, is_logging
    if not is_logging:
        return
    if len(msg.position) > 0:
        telemetry_buffer.append({
            'timestamp': time.time(),
            'position': list(msg.position),
            'velocity': list(msg.velocity),
            'torque': list(msg.effort) if len(msg.effort) > 0 else [0.0] * len(msg.position)
        })

# Bind telemetry subscriber directly to prevent thread conflicts
sub = wmx.create_subscription(JointState, '/joint_states', joint_states_callback, 10)

# 3. Smooth Half-Sine Profile Settings
duration = 4.0     # Total travel time (seconds)
rate_hz = 50       # Publishing rate (50Hz)
steps = int(duration * rate_hz)

print("⏳ Establishing connection on the DDS network...")
time.sleep(1.0)

print("🚀 [START] Initiating programmatic smooth drive and logging...")
start_time = time.time()

# 4. Generate smooth trajectory in real-time
for step in range(steps):
    t = step / rate_hz
    # Half-Sine scaling factor: s(t) smoothly ramps from 0.0 -> 1.0 -> 0.0
    s = 0.5 * (1.0 - np.cos(2.0 * np.pi * t / duration))
    
    # Apply scaled displacement to J1, J2, and J3 (6 arm joints + 2 gripper joints)
    j1_target = s * 0.8   # Joint 1 swing
    j2_target = s * -0.6  # Joint 2 dip
    j3_target = s * 0.5   # Joint 3 extend
    
    msg_positions = [j1_target, j2_target, j3_target, 0.0, 0.0, 0.0, 0.0, 0.0]
    
    cmd_msg = Float64MultiArray()
    cmd_msg.data = msg_positions
    pub_cmd.publish(cmd_msg)
    
    # Process callbacks (recording joint state feedbacks continuously)
    rclpy.spin_once(wmx, timeout_sec=1.0 / rate_hz)

# Capture trailing data for stabilization
is_logging = False
print(f"🛑 [FINISH] Motion finished. Total samples recorded: {len(telemetry_buffer)}")



## 📊 Step 2: Advanced Twin-X Axis Chart & Resource Cleanup
Now we load the captured telemetry buffer into a **Pandas DataFrame** for automated synchronization and plot the dual-axis dynamics chart. After displaying the plot, we will destroy the ROS 2 node to release device memory locks safely.


In [ ]:
import pandas as pd

# 🛠️ Programmatic Hotfix for Matplotlib 'RcParams' AttributeError
import matplotlib
if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = lambda key: matplotlib.rcParams.get(key)
import matplotlib.pyplot as plt

if len(telemetry_buffer) > 0:
    # 1. Load telemetry into Pandas DataFrame
    df = pd.DataFrame(telemetry_buffer)
    df['time_sec'] = df['timestamp'] - df['timestamp'].min()
    
    # 🎯 [CORRECTED] Extract only the Index 0 scalar (Joint 1) from the joint lists!
    df['j1_velocity'] = df['velocity'].apply(lambda v: v[0] if len(v) > 0 else 0.0)
    df['j1_torque'] = df['torque'].apply(lambda t: t[0] if len(t) > 0 else 0.0)
    
    # 2. Render Dual-Axis Dynamics Chart
    fig, ax1 = plt.subplots(figsize=(10, 4.5), dpi=100)
    
    # Primary Y-axis: Velocity (rad/s)
    color_vel = '#1f77b4'  # Deep professional blue
    ax1.set_xlabel('Time (seconds)', fontsize=11, fontweight='bold', labelpad=8)
    ax1.set_ylabel('Velocity (rad/s)', color=color_vel, fontsize=11, fontweight='bold')
    line1 = ax1.plot(df['time_sec'], df['j1_velocity'], color=color_vel, linewidth=2.5, label='Actual Velocity')
    ax1.tick_params(axis='y', labelcolor=color_vel)
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # Secondary Y-axis: Effort/Torque (Nm)
    ax2 = ax1.twinx()
    color_trq = '#d62728'  # High-visibility red
    ax2.set_ylabel('Torque / Effort (Nm)', color=color_trq, fontsize=11, fontweight='bold')
    line2 = ax2.plot(df['time_sec'], df['j1_torque'], color=color_trq, linewidth=2.0, linestyle='--', label='Motor Effort (Torque)')
    ax2.tick_params(axis='y', labelcolor=color_trq)
    
    # Combine Legends from both axes
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper right', frameon=True, shadow=True)
    
    plt.title('Joint 1 Velocity & Torque Correlation Analysis (Half-Sine Profile)', fontsize=13, fontweight='bold', pad=12)
    fig.tight_layout()
    plt.show()
    
    # 3. Clean up ROS 2 node resources safely to prevent lock errors
    wmx.destroy_node()
    print("🧹 [CLEANUP] ROS 2 node resources released safely. WMX Lock cleared.")
    
    # 4. Dynamics report
    print("\n📝 [DYNAMICS REPORT]")
    print("==========================================================================")
    print("- Smooth Acceleration: Notice the torque rising gently alongside velocity.")
    print("- Deceleration Braking: Effort mirrors velocity during deceleration.")
    print("- Constant Motion Stability: Ideal for tuning optimal motor capacities.")
    print("==========================================================================")
else:
    print("❌ No telemetry captured. Verify that Gazebo or WMX Driver is active.")


In [ ]:
# 4. Clean up ROS 2 node resources safely
if 'wmx' in locals():
    wmx.destroy_node()
print("🧹 [CLEANUP] ROS 2 node resources released safely. WMX Lock cleared.")